# AgroSmart · Fase 2 — Pipeline no Databricks

Este notebook executa no **Databricks Free Edition** o mesmo pipeline que roda localmente em
`ferramentas/executar_pipeline.py`, com o **mesmo código** (`src/pipeline/transformacoes_spark.py`):

```
upload do snapshot RAW (Volume)
  → leitura com schema explícito
  → Bronze  (tabela Delta, append, com procedência)
  → Silver  (normalização, validação, deduplicação + rejeitados com motivo)
  → Gold    (análises, laudos, agregados, qualidade)
  → consultas SQL
  → exportação do pacote processado (.zip) para o dashboard
```

**O que este notebook não faz:** não acessa o SQLite nem o `localhost` do seu computador, não
carrega TensorFlow nem o modelo Keras e não chama o Gemini. Ele só processa dados tabulares.
Usa apenas DataFrames e a sessão `spark` (sem RDD, sem `SparkContext`, sem DBFS legado).

### Antes de executar

1. **Código:** em *Workspace → Create → Git folder*, clone o repositório do AgroSmart. Abra este
   notebook a partir de `notebooks/` dentro dessa pasta — a célula de configuração adiciona a raiz do
   repositório ao `sys.path` para importar `src/pipeline`.
2. **Compute:** *Serverless* (padrão do Free Edition).
3. **Dados RAW:** no computador, gere o snapshot:
   `.venv\Scripts\python.exe ferramentas\exportar_snapshot.py --origem ambos`.
   Rode a célula de configuração uma vez (ela cria o Volume) e envie os dois arquivos de
   `dados/raw/<id_snapshot>/` (`manifesto.json` e `laudos.jsonl`) para
   `/Volumes/workspace/agrosmart/dados/raw/<id_snapshot>/` em *Catalog → workspace → agrosmart →
   dados → Upload to this volume*.
   *Atalho para demonstração:* ligue `GERAR_SIMULADOS = True` na seção 3 para gerar os dados
   simulados aqui mesmo, sem upload.

## 1. Configuração

In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

EM_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
CATALOGO, ESQUEMA, VOLUME = "workspace", "agrosmart", "dados"

# O notebook mora em notebooks/; a raiz do repositório é a pasta acima.
RAIZ_REPO = Path(os.environ.get("AGROSMART_REPO", Path(os.getcwd()).resolve().parent))
sys.path.insert(0, str(RAIZ_REPO))

if EM_DATABRICKS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUME}")
    BASE = Path(f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUME}")
else:
    # Conferência fora do Databricks (Spark local, sem Delta): as camadas viram views temporárias.
    from src.pipeline.transformacoes_spark import criar_sessao_local
    spark = criar_sessao_local()
    BASE = Path(os.environ.get("AGROSMART_NOTEBOOK_BASE", RAIZ_REPO / "dados"))
    display = lambda df: df.show(20, truncate=60)  # noqa: E731

PASTA_RAW = BASE / "raw"
PASTA_EXPORTACAO = BASE / "exportacoes"
PASTA_RAW.mkdir(parents=True, exist_ok=True)

# Datas do histórico da Fase 1 vêm sem fuso: são horário de Brasília.
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
print("Databricks:", EM_DATABRICKS, "| base de dados:", BASE)

In [ ]:
from pyspark.sql import functions as F

from src.pipeline import contratos, ingestao, normalizacao, pacote
from src.pipeline import transformacoes_spark as ts


def tabela(nome: str) -> str:
    return f"{CATALOGO}.{ESQUEMA}.{nome}" if EM_DATABRICKS else nome


def salvar(df, nome: str, modo: str = "overwrite"):
    """Grava a camada como tabela Delta (Databricks) e devolve a tabela relida."""
    if EM_DATABRICKS:
        escrita = df.write.format("delta").mode(modo)
        if modo == "append":
            escrita = escrita.option("mergeSchema", "true")
        else:
            escrita = escrita.option("overwriteSchema", "true")
        escrita.saveAsTable(tabela(nome))
    else:
        df.cache().createOrReplaceTempView(nome)
    return spark.table(tabela(nome))

## 2. Tabelas de normalização

As regras da Silver são dados, não código espalhado: sinônimos de cultura e de diagnóstico vindos
de `src/pipeline/normalizacao.py`. O CNN só emite as 12 classes de `src/rotulos.py`; o Gemini escreve
texto livre ("Míldio tardio", "Requeima (Phytophthora infestans)") — os dois caem em **Requeima**.

In [ ]:
display(spark.createDataFrame(normalizacao.tabela_diagnosticos(),
                              "sinonimo string, diagnostico string, categoria string"))

## 3. RAW — localizar e validar o snapshot

O snapshot traz `manifesto.json` (versão, origem, contagens, SHA-256) e `laudos.jsonl` (um laudo por
linha). A validação confere o hash: um arquivo truncado no upload é recusado aqui.

In [ ]:
GERAR_SIMULADOS = False  # True: gera os dados simulados aqui, sem upload
ID_SNAPSHOT = None       # None = o snapshot mais recente do Volume

if GERAR_SIMULADOS:
    sys.path.insert(0, str(RAIZ_REPO / "ferramentas"))
    import gerar_dados_simulados

    temporaria = Path("/tmp/agrosmart_raw")
    shutil.rmtree(temporaria, ignore_errors=True)
    registros = gerar_dados_simulados.gerar()
    gerado = ingestao.criar_snapshot(
        registros,
        fontes=[{"origem_dado": "simulado", "tipo": "gerador_no_databricks", "registros": len(registros),
                 "observacao": "dados sintéticos para demonstração — não são incidência real"}],
        destino=temporaria,
    )
    shutil.copytree(gerado, PASTA_RAW / gerado.name)
    ID_SNAPSHOT = gerado.name

snapshots = sorted(p.name for p in PASTA_RAW.iterdir() if (p / contratos.MANIFESTO).exists())
assert snapshots, f"Nenhum snapshot em {PASTA_RAW}. Faça o upload (ver topo) ou use GERAR_SIMULADOS = True."
pasta_snapshot = PASTA_RAW / (ID_SNAPSHOT or snapshots[-1])
manifesto_raw = ingestao.validar_snapshot(pasta_snapshot)
print(json.dumps({k: manifesto_raw[k] for k in ("id_snapshot", "extraido_em", "quantidade_registros",
                                                "registros_por_origem")}, indent=2, ensure_ascii=False))

## 4. Leitura com schema explícito

O schema vem do contrato (`contratos.CAMPOS_RAW`), não de inferência. Modo `PERMISSIVE`: uma linha
que não é JSON válido não derruba a leitura — ela segue com o texto original em `_registro_corrompido`.

In [ ]:
df_raw = ts.ler_raw(spark, str(pasta_snapshot / "laudos.jsonl"))
df_raw.printSchema()
display(df_raw.select("id_laudo", "origem_dado", "data_analise", "motor", "condicao",
                      "especie_original", "diagnostico_original").limit(10))

## 5. Bronze — os registros como chegaram, com procedência

Append numa tabela Delta: cada execução acrescenta seus registros marcados com `id_execucao`,
`processado_em` e `id_snapshot`. Nada é corrigido nem descartado aqui.

In [ ]:
ID_EXECUCAO = ingestao.novo_id("exec")
PROCESSADO_EM = ts.agora_utc()

bronze = ts.bronze(df_raw, ID_EXECUCAO, PROCESSADO_EM, persistir=False)
df_bronze = salvar(bronze, "bronze_laudos", modo="append").filter(F.col("id_execucao") == ID_EXECUCAO)
print(ID_EXECUCAO, "→", df_bronze.count(), "registros na Bronze")

## 6. Silver — normalização, validação e deduplicação

* datas ISO convertidas com `try_to_timestamp` (texto inválido vira rejeitado, não erro do job);
* condição, motor e origem reduzidos a códigos controlados;
* cultura e diagnóstico normalizados **sem apagar** `diagnostico_original`;
* `categoria_problema`: saudável, doença, praga, outra anomalia ou indeterminado;
* confiança numérica só no intervalo [0, 1] e **nunca** no laudo do VLM (a autoavaliação do Gemini
  não é probabilidade);
* identidade = `origem_dado + origem_instancia + id_laudo`: cópias idênticas ficam uma só (extração
  mais recente, desempate determinístico) e as demais vão para os rejeitados como `duplicado`; mesma
  identidade com conteúdo diferente → todas rejeitadas como `conflito_identidade_laudo`.

In [ ]:
silver, rejeitados = ts.silver(df_bronze)
df_silver = salvar(silver, "silver_laudos")
df_rejeitados = salvar(rejeitados, "silver_rejeitados")
print("válidos:", df_silver.count(), "| rejeitados:", df_rejeitados.count())

display(df_rejeitados.withColumn("motivo", F.explode("motivos")).groupBy("motivo").count()
        .orderBy(F.desc("count")))

In [ ]:
# Normalização do texto livre do VLM: original preservado, normalizado ao lado.
display(
    df_silver.filter((F.col("motor") == "vlm") & (F.col("condicao") == "doente"))
    .groupBy("diagnostico_original", "diagnostico_normalizado", "categoria_problema").count()
    .orderBy(F.desc("count"))
)

## 7. Gold

| Tabela | Unidade | Uso |
|---|---|---|
| `gold_analises` | uma linha por **análise** identificada | cards e gráficos principais |
| `gold_laudos` | todos os **laudos** válidos (inclusive reais sem id de análise) | motor, tabela de laudos |
| `gold_agregados` | semana × origem × UF × município × cultura | séries com **numerador e denominador** |
| `gold_qualidade` | métrica × origem | rejeitados, campos ausentes, identidade incompleta |

Numa análise do modo comparação (CNN + VLM), o **laudo principal** é o do CNN se ele foi conclusivo,
senão o do VLM — a mesma lógica do modo automático da Fase 1. Laudos sem `id_analise` (todos os reais:
a Fase 1 ainda não grava esse campo) não são agrupados por suposição: ficam fora de `gold_analises`.
Análises cujos laudos discordam no contexto (imagem, local, modo...) também ficam de fora e são
contadas em `gold_qualidade` como `analise_contexto_conflitante`.

In [ ]:
df_analises = salvar(ts.gold_analises(df_silver), "gold_analises")
df_laudos = salvar(ts.gold_laudos(df_silver), "gold_laudos")
df_agregados = salvar(ts.gold_agregados(df_analises), "gold_agregados")
df_qualidade = salvar(ts.gold_qualidade(df_bronze, df_silver, df_rejeitados, df_analises), "gold_qualidade")

for nome, df in [("gold_analises", df_analises), ("gold_laudos", df_laudos),
                 ("gold_agregados", df_agregados), ("gold_qualidade", df_qualidade)]:
    print(f"{nome:16s} {df.count():6d} linhas")

## 8. Consultas

Percentuais sempre como **soma dos numeradores ÷ soma dos denominadores** — nunca média de percentuais.

In [ ]:
# Condição das análises por origem (real × simulado nunca se misturam por padrão)
display(spark.sql(f"""
    SELECT origem_dado, condicao, COUNT(*) AS analises,
           ROUND(COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY origem_dado), 4) AS parcela
    FROM {tabela('gold_analises')}
    GROUP BY origem_dado, condicao
    ORDER BY origem_dado, analises DESC
"""))

In [ ]:
# Evolução semanal da parcela doente, a partir dos agregados
display(spark.sql(f"""
    SELECT semana, origem_dado,
           SUM(n_doente)   AS doentes,
           SUM(n_analises) AS analises,
           ROUND(SUM(n_doente) / SUM(n_analises), 4) AS parcela_doente
    FROM {tabela('gold_agregados')}
    GROUP BY semana, origem_dado
    ORDER BY origem_dado, semana
"""))

In [ ]:
# Diagnósticos mais frequentes entre as análises doentes
display(spark.sql(f"""
    SELECT diagnostico_normalizado, categoria_problema, COUNT(*) AS analises
    FROM {tabela('gold_analises')}
    WHERE condicao = 'doente'
    GROUP BY diagnostico_normalizado, categoria_problema
    ORDER BY analises DESC
    LIMIT 10
"""))

In [ ]:
# Município × cultura com maior parcela doente (mínimo de 20 análises)
display(spark.sql(f"""
    SELECT uf, municipio, cultura, SUM(n_doente) AS doentes, SUM(n_analises) AS analises,
           ROUND(SUM(n_doente) / SUM(n_analises), 4) AS parcela_doente
    FROM {tabela('gold_agregados')}
    WHERE municipio IS NOT NULL
    GROUP BY uf, municipio, cultura
    HAVING SUM(n_analises) >= 20
    ORDER BY parcela_doente DESC
"""))

In [ ]:
# Modo comparação: quantas vezes CNN e VLM concordaram na condição
display(spark.sql(f"""
    SELECT condicao_cnn, condicao_vlm, concordancia_motores, COUNT(*) AS analises
    FROM {tabela('gold_analises')}
    WHERE comparacao
    GROUP BY condicao_cnn, condicao_vlm, concordancia_motores
    ORDER BY analises DESC
"""))

In [ ]:
# Qualidade: rejeitados e lacunas
display(
    df_qualidade.filter(F.col("metrica").startswith("rejeitado") | F.col("metrica").isin(
        "localidade_ausente", "identidade_incompleta", "laudo_sem_analise", "diagnostico_nao_mapeado"))
    .withColumn("parcela", F.round(F.col("valor") / F.col("denominador"), 4))
    .orderBy("origem_dado", "metrica")
)

## 9. Exportação do pacote processado

O pacote é montado em `/tmp` (disco local do driver), validado e compactado; o `.zip` vai para
`/Volumes/workspace/agrosmart/dados/exportacoes/`. Baixe-o pelo *Catalog Explorer* e, no computador:

```
.venv\Scripts\python.exe ferramentas\importar_processados.py exec_....zip
```

O dashboard passa a mostrar esta execução assim que a página for recarregada.

In [ ]:
temporaria = Path("/tmp/agrosmart_processados") if EM_DATABRICKS else BASE / "tmp_processados"
shutil.rmtree(temporaria, ignore_errors=True)

pasta_pacote = ts.exportar_pacote(
    df_analises, df_laudos, df_agregados, df_qualidade, df_rejeitados,
    id_execucao=ID_EXECUCAO,
    processado_em=PROCESSADO_EM,
    metadados={
        "ambiente": "databricks" if EM_DATABRICKS else "spark-local (notebook)",
        "versao_spark": spark.version,
        "id_snapshot": manifesto_raw["id_snapshot"],
        "snapshot_extraido_em": manifesto_raw["extraido_em"],
        "registros_raw_por_origem": manifesto_raw["registros_por_origem"],
    },
    destino=temporaria,
    marcar_como_atual=False,
)
manifesto = pacote.validar(pasta_pacote)

arquivo_zip = shutil.make_archive(str(temporaria / ID_EXECUCAO), "zip", root_dir=temporaria, base_dir=ID_EXECUCAO)
PASTA_EXPORTACAO.mkdir(parents=True, exist_ok=True)
destino_zip = PASTA_EXPORTACAO / f"{ID_EXECUCAO}.zip"
shutil.copyfile(arquivo_zip, destino_zip)
print("Pacote validado:", manifesto["id_execucao"], manifesto["por_origem"])
print("Baixe:", destino_zip)